# PromSec

## Setup

Set `OPENAI_API_KEY` and `PROMSEC_LLM_MODEL`, and `PROMSEC_JUDGE_MODEL` for LAAJ.

In [ ]:
from pathlib import Path
import os
import json
import shutil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib.ticker import MaxNLocator

from utils import (
    NODE_FEATURE_DIM,
    cfg_to_pyg_data,
    generate_cfg_from_code,
    load_original_code,
)
from promsec.config import load_config
from promsec.io import atomic_write_json, sha256_text
from promsec.llm import LLMClient
from promsec.model import Discriminator, Generator
from promsec.security import scan_cwes
from promsec.runner import run_iteration as promsec_iteration
from promsec.train import train_models

ROOT = Path.cwd()
TRAIN_DIR = ROOT / "data" / "training"
TEST_DIR = ROOT / "data" / "test"
MODEL_DIR = ROOT / "models"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)
CONFIG = load_config("configs/default.yaml", root=ROOT)

PROMSEC_LLM_MODEL = os.environ.get("PROMSEC_LLM_MODEL")

## Similarity utilities

In [ ]:
import ast


def compare_asts(tree1, tree2):
    nodes1 = list(ast.walk(tree1))
    nodes2 = list(ast.walk(tree2))
    if not nodes1 and not nodes2:
        return 1.0
    common_nodes = sum(
        1 for node in nodes1 if any(isinstance(node, type(other)) for other in nodes2)
    )
    return common_nodes / max(len(nodes1), len(nodes2), 1)


def calc_ast_similarity(code1, code2):
    try:
        return compare_asts(ast.parse(code1), ast.parse(code2))
    except SyntaxError:
        return 0.0

## Training data

In [ ]:
training_files = sorted(TRAIN_DIR.glob("*.py"))
if not training_files:
    raise RuntimeError(f"No training programs found in {TRAIN_DIR}")

training_graphs = []
for path in training_files:
    cfg = generate_cfg_from_code(path)
    if cfg.number_of_nodes() == 0:
        raise RuntimeError(f"Empty training graph: {path.name}")
    pyg_data = cfg_to_pyg_data(cfg)
    if pyg_data.x.shape != (cfg.number_of_nodes(), NODE_FEATURE_DIM):
        raise RuntimeError(f"Feature shape mismatch: {path.name}")
    if pyg_data.raw_features.shape != (cfg.number_of_nodes(), 4):
        raise RuntimeError(f"Raw-feature shape mismatch: {path.name}")
    if pyg_data.edge_index.numel() and int(pyg_data.edge_index.max()) >= cfg.number_of_nodes():
        raise RuntimeError(f"Edge index out of range: {path.name}")
    training_graphs.append(pyg_data)

print(f"Prepared {len(training_graphs)} training graphs with {NODE_FEATURE_DIM} features per node.")

## Models and training

Set `TRAIN_FROM_SCRATCH = 1` to train a fresh gGAN; `0` uses the packaged checkpoints.

In [ ]:
TRAIN_FROM_SCRATCH = 0  # 0 = packaged checkpoints; 1 = train a fresh gGAN

if TRAIN_FROM_SCRATCH == 1:
    training_result = train_models(CONFIG)
    print(json.dumps({key: training_result[key] for key in ("epochs", "samples", "generator", "discriminator")}, indent=2))
    training_log = pd.read_json(ROOT / CONFIG["run"]["root"] / CONFIG["run"]["name"] / "training" / "training.jsonl", lines=True)
    display(training_log)

## Evaluation data and checkpoints

In [ ]:
test_files = sorted(TEST_DIR.glob("*.py"))
if not test_files:
    raise RuntimeError(f"No evaluation programs found in {TEST_DIR}")

test_graphs = {}
for path in test_files:
    cfg = generate_cfg_from_code(path)
    if cfg.number_of_nodes() == 0:
        raise RuntimeError(f"Empty evaluation graph: {path.name}")
    pyg_data = cfg_to_pyg_data(cfg)
    if pyg_data.x.shape != (cfg.number_of_nodes(), NODE_FEATURE_DIM):
        raise RuntimeError(f"Feature shape mismatch: {path.name}")
    if pyg_data.raw_features.shape != (cfg.number_of_nodes(), 4):
        raise RuntimeError(f"Raw-feature shape mismatch: {path.name}")
    test_graphs[path.name] = pyg_data

print(f"Prepared {len(test_graphs)} evaluation graphs.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generator = Generator().to(device)
discriminator = Discriminator().to(device)

generator_checkpoint = MODEL_DIR / ("fresh_generator_cfg.pt" if TRAIN_FROM_SCRATCH == 1 else "trained_generator_cfg.pt")
discriminator_checkpoint = MODEL_DIR / ("fresh_discriminator_cfg.pt" if TRAIN_FROM_SCRATCH == 1 else "trained_discriminator_cfg.pt")

generator.load_state_dict(
    torch.load(generator_checkpoint, map_location=device, weights_only=True)
)
discriminator.load_state_dict(
    torch.load(discriminator_checkpoint, map_location=device, weights_only=True)
)
generator.eval()
discriminator.eval()
print(f"Loaded checkpoints on {device}: {generator_checkpoint.name}, {discriminator_checkpoint.name}")

## Security enforcement

Requires Bandit and LLM access; disabled by default.

In [ ]:
def make_llm(transport=None):
    """LLM client for enforcement: OpenAI-compatible endpoint, cached responses."""
    if not PROMSEC_LLM_MODEL:
        raise RuntimeError("Set PROMSEC_LLM_MODEL before running security enforcement.")
    return LLMClient(
        PROMSEC_LLM_MODEL,
        base_url=os.environ.get("PROMSEC_LLM_BASE_URL") or None,
        cache_path=RESULTS_DIR / "llm_cache.sqlite",
        transport=transport,
    )


def run_iteration(ci, llm):
    """One PromSec iteration on code ci; returns the new code and the iteration diagnostics."""
    guidance = CONFIG["guidance"]
    return promsec_iteration(
        ci, generator, llm, device,
        syntax_repair_attempts=int(CONFIG["llm"]["syntax_repair_attempts"]),
        top_k=int(guidance["top_k"]),
        min_abs_change=float(guidance["min_abs_change"]),
    )


RUN_ENFORCEMENT_EVALUATION = False
MAX_SECURITY_ITERATIONS = 20
EVALUATION_FILES = sorted(test_graphs)

security_results = []
security_df = pd.DataFrame()
OUTPUT_DIR = RESULTS_DIR / "vetted_code"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if RUN_ENFORCEMENT_EVALUATION:
    if shutil.which("bandit") is None:
        raise RuntimeError("Bandit is required for security evaluation. Install the dependencies in requirements.txt.")
    if not os.environ.get("OPENAI_API_KEY"):
        raise RuntimeError("Set OPENAI_API_KEY before running security enforcement.")
    if not PROMSEC_LLM_MODEL:
        raise RuntimeError("Set PROMSEC_LLM_MODEL before running security enforcement.")

    llm = make_llm()
    with torch.no_grad():
        for filename in EVALUATION_FILES:
            original_path = TEST_DIR / filename
            original_code = load_original_code(original_path)
            current_code = original_code

            before_count = after_count = None
            before_cwes = after_cwes = []
            feature_change = 0.0
            ast_similarity = 1.0
            iterations_used = 0
            status = "failed"
            error = ""

            candidate_path = OUTPUT_DIR / filename.replace(".py", "_vet.py")
            candidate_path.unlink(missing_ok=True)
            try:
                before_count, before_cwes = scan_cwes(original_code, require_compile=False)
                after_count, after_cwes = before_count, before_cwes
                status = "already_clean" if before_count == 0 else "completed"
                for iteration in range(MAX_SECURITY_ITERATIONS):
                    if after_count == 0:
                        break
                    step = run_iteration(current_code, llm)
                    current_code = step["code"]
                    feature_change = step["feature_change"]
                    after_count, after_cwes = scan_cwes(current_code)
                    ast_similarity = calc_ast_similarity(original_code, current_code)
                    iterations_used = iteration + 1
                if before_count > 0:
                    status = "secured" if after_count == 0 else "max_iterations_reached"
            except Exception as exc:
                status = "failed"
                error = f"{type(exc).__name__}: {exc}"
                after_count = None

            (OUTPUT_DIR / filename).write_text(original_code, encoding="utf-8")
            if status != "failed":
                candidate_path.write_text(current_code, encoding="utf-8")
            atomic_write_json(RESULTS_DIR / "samples" / (Path(filename).stem + ".json"), {
                "file": filename, "status": status, "source_hash": sha256_text(original_code),
                "candidate_hash": sha256_text(current_code) if status != "failed" else None,
                "error": error,
            })

            security_results.append(
                {
                    "file": filename,
                    "status": status,
                    "error": error,
                    "cwe_count_before": before_count,
                    "cwe_count_after": after_count,
                    "cwes_before": ",".join(map(str, before_cwes)),
                    "cwes_after": ",".join(map(str, after_cwes)),
                    "cwe_types_net_reduction": before_count - after_count if before_count is not None and after_count is not None else None,
                    "iterations": iterations_used,
                    "gnn_feature_change": feature_change,
                    "ast_similarity": ast_similarity,
                }
            )

    security_df = pd.DataFrame(security_results)
    security_df.to_csv(RESULTS_DIR / "security_evaluation.csv", index=False)
    display(security_df)
else:
    security_df = pd.DataFrame()
    print("Security-enforcement evaluation is disabled. Set RUN_ENFORCEMENT_EVALUATION=True to run it.")

## LAAJ functional similarity

In [ ]:
from evaluation.functional_similarity import (
    extract_behavioral_contract,
    make_openai_json_completion,
    score_candidate_functionality,
)

RUN_LAAJ_EVALUATION = False
PROMSEC_JUDGE_MODEL = os.environ.get("PROMSEC_JUDGE_MODEL")
laaj_results = []
laaj_df = pd.DataFrame()

if RUN_LAAJ_EVALUATION:
    if not PROMSEC_JUDGE_MODEL:
        raise RuntimeError("Set PROMSEC_JUDGE_MODEL before running LAAJ evaluation.")
    (RESULTS_DIR / "functional_similarity.csv").unlink(missing_ok=True)
    judge_json = make_openai_json_completion(PROMSEC_JUDGE_MODEL)

    for filename in EVALUATION_FILES:
        original_path = TEST_DIR / filename
        vetted_path = OUTPUT_DIR / filename.replace(".py", "_vet.py")
        if not vetted_path.exists():
            continue

        original_code = load_original_code(original_path)
        vetted_code = load_original_code(vetted_path)
        record_path = RESULTS_DIR / "samples" / (Path(filename).stem + ".json")
        if not record_path.exists():
            continue
        record = json.loads(record_path.read_text(encoding="utf-8"))
        if (record.get("status") == "failed" or
                record.get("source_hash") != sha256_text(original_code) or
                record.get("candidate_hash") != sha256_text(vetted_code)):
            continue
        contract = extract_behavioral_contract(original_code, judge_json)
        report = score_candidate_functionality(contract, vetted_code, judge_json)
        row = report.to_dict()
        row.update(source_hash=record["source_hash"], candidate_hash=record["candidate_hash"], judge_model=PROMSEC_JUDGE_MODEL, judge_endpoint=os.environ.get("PROMSEC_JUDGE_BASE_URL") or "")
        dimensions = row.pop("dimensions")
        row.update({f"dimension_{name}": score for name, score in dimensions.items()})
        laaj_results.append({"file": filename, **row})

    laaj_df = pd.DataFrame(laaj_results)
    laaj_df.to_csv(RESULTS_DIR / "functional_similarity.csv", index=False)
    display(laaj_df)
    if not laaj_df.empty:
        print(f"Mean LAAJ functional similarity: {laaj_df['laaj_score'].mean():.4f}")
        print(f"Mean gated functional score: {laaj_df['gated_score'].mean():.4f}")
else:
    laaj_df = pd.DataFrame()
    print("LAAJ evaluation is disabled. Set RUN_LAAJ_EVALUATION=True to run it.")

## Results

In [ ]:
if not security_df.empty:
    completed = security_df[security_df["status"] != "failed"].copy()
    vulnerable = security_df[security_df["cwe_count_before"] > 0]
    summary = {
        "programs_evaluated": int(len(security_df)),
        "programs_completed": int(len(completed)),
        "vulnerable_programs_evaluated": int(len(vulnerable)),
        "mean_cwe_types_before": float(completed["cwe_count_before"].mean()) if not completed.empty else np.nan,
        "mean_cwe_types_after": float(completed["cwe_count_after"].mean()) if not completed.empty else np.nan,
        "clean_after_rate": float((completed["cwe_count_after"] == 0).mean()) if not completed.empty else np.nan,
        "enforcement_success_rate_on_vulnerable": float((vulnerable["cwe_count_after"] == 0).mean()) if not vulnerable.empty else np.nan,
        "mean_gnn_feature_change": float(completed["gnn_feature_change"].mean()) if not completed.empty else np.nan,
        "mean_ast_similarity": float(completed["ast_similarity"].mean()) if not completed.empty else np.nan,
    }
    current_laaj = []
    if not laaj_df.empty:
        for _, row in laaj_df.iterrows():
            candidate = OUTPUT_DIR / row["file"].replace(".py", "_vet.py")
            source = TEST_DIR / row["file"]
            if (row["file"] in set(completed["file"]) and source.is_file() and candidate.is_file()
                    and row.get("source_hash") == sha256_text(load_original_code(source))
                    and row.get("candidate_hash") == sha256_text(load_original_code(candidate))
                    and row.get("judge_model") == os.environ.get("PROMSEC_JUDGE_MODEL")
                    and row.get("judge_endpoint", "") == (os.environ.get("PROMSEC_JUDGE_BASE_URL") or "")):
                current_laaj.append(row)
    if current_laaj:
        verified_laaj = pd.DataFrame(current_laaj)
        summary["mean_laaj_similarity"] = float(verified_laaj["laaj_score"].mean())
        summary["mean_laaj_gated_score"] = float(verified_laaj["gated_score"].mean())

    summary_df = pd.DataFrame([summary])
    summary_df.to_csv(RESULTS_DIR / "summary.csv", index=False)
    display(summary_df)

    x = np.arange(len(completed))
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.scatter(x, completed["cwe_count_before"], label="Original")
    ax.scatter(x, completed["cwe_count_after"], label="PromSec")
    ax.set_xlabel("Program index")
    ax.set_ylabel("Unique CWE types")
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.legend()
    ax.grid(True, linestyle="--", linewidth=0.5)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / "cwe_counts.png", dpi=150)
    plt.show()